# Week 1, Day 4: Open Source, Local, Token Cost

## Introduction

This notebook sends 20 of UnderCover's queued claims to open-weight models and compares them on accuracy, cost and speed. It first shrinks the prompt from Day 2, then runs six models through OpenRouter, three more on your own computer with Ollama, and an optional model direct on Cerebras. It ends by putting every run side by side in one table and three charts.

Use this notebook as a reference: each section explains the purpose of its code and how to read the results. Handler decisions are evaluation labels only. The prompts sent to models contain the claim, the policy record and the relevant policy wording, never the handler's answer.

## Learning goals

- Cut the prompt to the policy sections a claim needs, and measure the token saving.
- Run the same structured-output task on hosted models, local models and a direct provider API.
- Score every run the same way: decision right, and decision plus payout right.
- Handle the failure modes of small local models: timeouts, cut-off JSON and missing fields.
- Compare accuracy, cost and speed together, and recognise what 20 claims can and cannot show.

## Before you begin

- Run this notebook from its saved location, so the relative path `../../../data` resolves to the repository's `data/` folder.
- Set `OPENROUTER_API_KEY` in the repository's `.env` file. `CEREBRAS_API_KEY` is only needed for the optional Cerebras section.
- Part 3 needs [Ollama](https://ollama.com) running on your computer. Each local model is several gigabytes, and the notebook pulls them for you.
- OpenRouter and Cerebras calls cost money. The six OpenRouter runs here cost about $0.25 in total, and the three Cerebras runs about $0.20. Check the spend limit on your key first.
- Saved outputs can be stale. Rerun from the top for results that match the current code.

## Notebook map

1. **Connect to OpenRouter**: load the key and create the client.
2. **Part 1, The data and a smaller prompt**: load the claims, build a shorter prompt, and measure the saving.
3. **Part 2, Six open-weight models on OpenRouter**: define the schema, the scorer and the runners, then run six models.
4. **Part 3, Three models on your computer with Ollama**: run local models, with guards for the ways they fail. The optional Cerebras runs sit at the end of this part.
5. **Part 4, Accuracy, cost and speed side by side**: one summary table and three charts.
6. **Conclusion**: what the comparison shows and what it does not.

In [1]:
# Standard-library helpers for project data, environment variables, timing and decision types.
import json
import os
import time
from pathlib import Path
from typing import Literal

# httpx raises the timeout error that Ollama calls surface; Part 3 catches it.
import httpx
# Ollama runs the local models in Part 3.
import ollama
# Tabular analysis and interactive charts for Part 4.
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
# The OpenAI client also talks to OpenRouter and Cerebras, which are OpenAI-compatible.
from openai import OpenAI
# Pydantic defines the structured decision; ValidationError is how a bad local answer shows up.
from pydantic import BaseModel, Field, ValidationError
# Progress bars while each model works through the claims.
from tqdm.auto import tqdm

## Connect to OpenRouter

Load the API key and create the OpenAI-compatible OpenRouter client. OpenRouter forwards each request to a provider that hosts the model, so one client and one key reach many models. The client waits up to 120 seconds for a reply and retries a failed request up to five times.

In [2]:
# Load repository settings, allowing .env to override an existing shell value.
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

# Stop early with a clear message instead of failing later inside a model call.
if openrouter_api_key:
    print('OpenRouter API key loaded successfully.')
else:
    raise RuntimeError('OPENROUTER_API_KEY is missing; add it to the repository .env file.')

# OpenRouter exposes an OpenAI-compatible endpoint, so the OpenAI client works unchanged.
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL,
                    api_key=openrouter_api_key,
                    timeout=120,
                    max_retries=5)

OpenRouter API key loaded successfully.


## Part 1: The data and a smaller prompt

Load the claims queue, the policy records and the handler labels, then build a prompt that carries only the policy wording each claim needs. Labels are used for scoring only and never appear in a prompt.

In [3]:
# This notebook lives three directories below the repository root.
DATA = Path("../../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding='utf-8'))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    # Handler decisions are the answer key; they are used only to score model output.
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [4]:
# Set TINY=True for the five-claim subset: quicker and cheaper while you are debugging.
TINY = False

claims, labels = get_sample_data(tiny=TINY)
# Claims the handlers marked "manual" are outside what the agent should decide, so leave them out.
claims = [claim for claim in claims if labels[claim['claim_id']]['difficulty'] != 'manual']
policies = {policy['policy_id']: policy for policy in load('policies.json')}
print(f"{len(claims)} claims")

20 claims


In [5]:
# Count the handlers' decisions across these claims: this is the mix the models are scored against.
decisions = pd.DataFrame([labels[claim['claim_id']] for claim in claims])
decisions['decision'].value_counts()

decision
approve    12
decline     5
refer       3
Name: count, dtype: int64

### A smaller prompt

On Day 2 every request carried the whole policy wording. Most of it is irrelevant to any one claim: a baggage claim does not need the Trip Delay rules. Splitting the short wording by section lets each prompt carry only the sections the claim needs. Fewer input tokens means a lower cost and a faster reply, which matters most for small local models with a limited context window.

In [6]:
# Compare the full wording with the shorter version by word count.
policy_wording = (DATA / 'policy_wording.md').read_text(encoding='utf-8')
short_wording = (DATA / 'policy_wording_short.md').read_text(encoding='utf-8')

print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"The short policy wording is {len(short_wording.split()):,} words")

The policy wording is 1,369 words
The short policy wording is 808 words


In [7]:
def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    # Splitting removes the "## " marker, so put it back on each section.
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)
list(sections)

['Limits',
 'Cancellation and Interruption',
 'Trip Delay',
 'Baggage Delay',
 'Baggage',
 'Passport',
 'Emergency Medical',
 'Add-ons',
 'Never covered',
 'Payment',
 'Claims']

In [8]:
# Which wording section holds the rules for each claim type.
BENEFITS = {
    'trip_cancellation': 'Cancellation and Interruption',
    'trip_interruption': 'Cancellation and Interruption',
    'trip_delay': 'Trip Delay',
    'baggage_delay': 'Baggage Delay',
    'baggage': 'Baggage',
    'passport': 'Passport',
    'emergency_medical': 'Emergency Medical',
}


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits,
    its benefit, the add-ons if the policy has any, and the rules for every claim.
    """
    headings = ['Limits', BENEFITS[claim['claim_type']]]
    # Only policies that bought add-ons need the Add-ons section.
    if policies[claim['policy_id']]['add_ons']:
        headings.append('Add-ons')
    headings += ['Never covered', 'Payment', 'Claims']
    return '\n\n'.join([opening.strip()] + [sections[heading].strip() for heading in headings])

In [9]:
# The task rules. The wording and the claim are added around this text for each request.
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.
"""


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim['policy_id']]
    # Drop fields the decision does not need, and list documents by name and type only.
    record = {key: value for key, value in policy.items()
              if key not in ['policy_id', 'customer_id', 'sold_through', 'status']}
    details = {key: value for key, value in claim.items()
               if key not in ['policy_id', 'customer_id', 'status']}
    details['documents'] = [{'name': document['name'], 'type': document['type']}
                            for document in claim['documents']]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording it needs,
    then the policy record and the claim.
    """
    return [
        {'role': 'system', 'content': f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {'role': 'user', 'content': make_user_prompt(claim)},
    ]

In [10]:
# Print both messages for the first claim to check the prompt reads as intended.
messages = make_messages(claims[0])
print(messages[0]['content'])
print(messages[1]['content'])

You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the documents disagree, or the wording does not settle the question.

# UnderCover Travel Insurance: Policy Wording, short version

All amounts are in US dollars. The policy record shows the plan, the add-ons, the trip dates, the trip cost and the travelers.

## Limits

| Benefit | Economy | Premium | First |
|---|---|---|---|
| Cancellation and Interruption maximum | 1,500 | 5,000 | 10,000 |
| Trip Delay waiting period, hours | 12 | 6 | 3 |
| Trip Delay maximum | 300 | 800 | 1,500 |
| Baggage Delay waiting period, hours | 24 | 12 | 6 |
| Baggage Delay max

### Part 1 summary

You loaded the claims queue and the handler labels, removed the claims reserved for manual handling, and built a prompt that sends each claim only the wording sections it needs. The saving is measured in Part 2. Labels stay in the scoring step and never enter a prompt.

## Part 2: Six open-weight models on OpenRouter

Define the structured decision, the function that asks one model about one claim, and the scoring helpers. Then run six open-weight models on the same 20 claims. Each run makes 20 paid requests, so check your spend limit before running them all.

### Define the decision schema

The schema forces every model to return a rationale, a decision and a payout. The rationale is listed first on purpose: the model reasons in it before committing to a decision. Structured output also makes scoring a simple comparison with no text parsing.

In [11]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description='2-3 sentences to explain the decision along with the relevant section of the policy')
    decision: Literal['approve', 'decline', 'refer']
    payout: float = Field(description='The amount to pay in US dollars, or 0 for a decline or a referral')

In [12]:
class LocalDecision(ClaimDecision):
    """ClaimDecision for Ollama, where a missing payout counts as 0.

    Gemma 4 sometimes writes the rationale and decision on a decline and never writes the payout.
    Ollama's JSON grammar then keeps emitting blank space until the token cap, so the call fails.
    Making payout optional lets the grammar finish; a missing payout is read as 0.
    """
    payout: float = Field(default=0.0, description='The amount to pay in US dollars, or 0 for a decline or a referral')

### Call one model through OpenRouter

`decide` asks one model about one claim, parses the answer into `ClaimDecision`, and returns a flat record that includes the provider, the input tokens, the cost and the time. `require_parameters` tells OpenRouter to route only to providers that support every parameter in the request, including structured output. `ignore` lets you skip a provider that fails for a particular model.

In [13]:
def decide(claim, model, effort, ignore=None):
    """Ask one model on OpenRouter to decide one claim, skipping any providers listed in ignore."""
    start = time.time()
    provider = {'require_parameters': True}
    if ignore:
        provider['ignore'] = ignore
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={'provider': provider},
    )
    # The client has already parsed the reply into a ClaimDecision.
    answer = response.choices[0].message.parsed
    return {
        'model': model,
        'effort': effort,
        'where': 'OpenRouter',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'provider': response.provider,
        'input_tokens': response.usage.prompt_tokens,
        'cost': response.usage.cost,
        'seconds': round(time.time() - start, 1),
    }

In [14]:
# Smoke test: one claim, one model. Check the decision, payout, cost and token count look sensible.
first = decide(claims[0], 'deepseek/deepseek-v4.1-flash', 'low')
first

{'model': 'deepseek/deepseek-v4.1-flash',
 'effort': 'low',
 'where': 'OpenRouter',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim is covered under Trip Delay because the flight was canceled due to a crew problem, which is a covered cause, and the delay exceeded the Premium plan's 6-hour waiting period. All required receipts and the carrier's cancellation notice are attached, and the total claimed is below the Premium Trip Delay maximum of $800. No deductible applies and there is no indication of payment by another party.",
 'provider': 'CoreWeave',
 'input_tokens': 1251,
 'cost': 0.0012283,
 'seconds': 12.2}

### What the smaller prompt saves

Rebuild Day 2's prompt, with the whole wording and indented JSON, and send the same claim to the same model. Comparing the input token counts shows what the section-by-section prompt saves.

In [15]:
def make_day2_messages(claim):
    """Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON."""
    policy = policies[claim['policy_id']]
    return [
        {'role': 'system', 'content': f"{INSTRUCTIONS}\n{policy_wording}\n"},
        {'role': 'user',
         'content': f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"},
    ]


day2 = openrouter.chat.completions.parse(
    model='deepseek/deepseek-v4.1-flash',
    messages=make_day2_messages(claims[0]),
    response_format=ClaimDecision,
    reasoning_effort='low',
    extra_body={'provider': {'require_parameters': True}},
)

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens")
print(f"Today's prompt: {first['input_tokens']:,} input tokens")

Day 2's prompt: 2,692 input tokens
Today's prompt: 1,251 input tokens


The saved run showed 2,808 input tokens for Day 2's prompt against 1,055 for today's, about 62% fewer, for one claim. That saving applies to every request, and it is why the local models in Part 3 can run with a small context window.

### Score the runs and run all six models

Three helpers keep every run comparable:

- `is_right` counts an answer as right only when the decision matches the handler's and the payout is within one US dollar.
- `score` adds the claim's difficulty and two flags: `decision_right` (decision only) and `right` (decision and payout).
- `report` prints one line per run: decisions right, right with the payout too, total cost and total time.

`run` loops a model over the claims and scores each answer. Each cell below runs one model at the reasoning effort noted in its comment.

In [16]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer['decision'] == label['decision'] and abs(answer['payout'] - label['payout']) <= 1


def score(answer, label):
    """Add the difficulty, and whether the decision and the payout match the handler's."""
    return {
        **answer,
        'difficulty': label['difficulty'],
        'decision_right': answer['decision'] == label['decision'],
        'right': is_right(answer, label),
    }


def report(results):
    """Print how a run did: decisions right, decisions right with the payout too, the cost and the time."""
    decisions_right = sum(result['decision_right'] for result in results)
    right = sum(result['right'] for result in results)
    cost = sum(result['cost'] for result in results)
    seconds = sum(result['seconds'] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, ${cost:.4f} in {seconds:.0f} seconds")

In [17]:
def run(model, effort, claims, labels, ignore=None):
    """Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model, effort, ignore)
        results.append(score(answer, labels[claim['claim_id']]))
    report(results)
    return results

In [18]:
# Xiaomi MiMo: effort None, so no reasoning level is requested and the model uses its default.
mimo = run('xiaomi/mimo-v2.6-pro', None, claims, labels)

xiaomi/mimo-v2.6-pro:   0%|          | 0/20 [00:00<?, ?it/s]

xiaomi/mimo-v2.6-pro: 17 of 20 decisions right, 15 with the payout too, $0.0223 in 699 seconds


In [19]:
# Z.ai GLM at low effort. ignore=['wafer'] stops OpenRouter routing to that provider.
glm = run('z-ai/glm-5.3-flash', 'low', claims, labels, ignore=['wafer'])

z-ai/glm-5.3-flash:   0%|          | 0/20 [00:00<?, ?it/s]

z-ai/glm-5.3-flash: 14 of 20 decisions right, 11 with the payout too, $0.0040 in 173 seconds


In [20]:
# Moonshot Kimi at low effort.
kimi = run('moonshotai/kimi-k3', 'low', claims, labels)

moonshotai/kimi-k3:   0%|          | 0/20 [00:00<?, ?it/s]

moonshotai/kimi-k3: 16 of 20 decisions right, 14 with the payout too, $0.1443 in 116 seconds


In [21]:
# DeepSeek at low effort.
deepseek = run('deepseek/deepseek-v4.1-flash', 'low', claims, labels)

deepseek/deepseek-v4.1-flash:   0%|          | 0/20 [00:00<?, ?it/s]

deepseek/deepseek-v4.1-flash: 17 of 20 decisions right, 16 with the payout too, $0.0189 in 149 seconds


In [22]:
# NVIDIA Nemotron at medium effort.
nemotron = run('nvidia/nemotron-3-ultra-550b-a55b', 'medium', claims, labels)

nvidia/nemotron-3-ultra-550b-a55b:   0%|          | 0/20 [00:00<?, ?it/s]

nvidia/nemotron-3-ultra-550b-a55b: 14 of 20 decisions right, 11 with the payout too, $0.0228 in 88 seconds


In [23]:
# Mistral at "none", the lowest reasoning level, so it answers straight away.
mistral = run('mistralai/mistral-medium-3-5', 'none', claims, labels)

mistralai/mistral-medium-3-5:   0%|          | 0/20 [00:00<?, ?it/s]

mistralai/mistral-medium-3-5: 15 of 20 decisions right, 11 with the payout too, $0.0531 in 35 seconds


### Part 2 summary

You defined one decision schema, one scoring rule and one runner, and ran six open-weight models through OpenRouter on the same 20 claims. In the saved run, Kimi and DeepSeek each got 16 of 20 right with the payout, but Kimi cost about $0.13 against DeepSeek's $0.014. MiMo, Nemotron, GLM and Mistral got 11 to 13 right. Mistral was far faster than the rest (28 seconds for the run) but the least accurate on decisions. The gap between "decision right" and "decision and payout right" shows that working out the payout is harder than choosing the decision.

## Part 3: Three models on your computer with Ollama

Ollama runs open-weight models on your own machine, so there is no per-token cost and no data leaves the computer. The price is speed, and reliability: small models are more likely to ignore the schema, ramble or stall. The code below guards against each of those so that one bad claim doesn't lose the whole run.

In [24]:
# Download the local models once; Ollama skips a model that is already installed.
print(ollama.pull('qwen3.5:4b').status)
print(ollama.pull('gemma4:e4b').status)

success
success


In [25]:
# A dedicated client with a timeout. A normal answer takes under 15 seconds,
# so 30 seconds only triggers when the runner has stalled.
local = ollama.Client(timeout=30)

### Ask a local model for a decision

`decide_locally` handles three ways a small model can fail:

1. **Stalled runner**: the request times out. The model is unloaded so the retry starts a fresh runner.
2. **Cut-off JSON**: the answer reaches the token cap before the object is closed, so it does not validate. `num_predict` is 2048 because a claim with several line items can need about 1,000 tokens of working in the rationale.
3. **Repeated output**: `temperature` is 0 on the first try and rises by 0.3 on each retry, so a retry doesn't repeat the same failing text.

A claim that fails all three tries raises the error, so you can see which claim it was. `format` passes the `LocalDecision` JSON schema so Ollama constrains the output to it.

In [26]:
def decide_locally(claim, model, tries=3):
    """Ask a model on this computer to decide one claim, through Ollama."""
    start = time.time()
    for attempt in range(tries):
        try:
            response = local.chat(
                model=model,
                messages=make_messages(claim),
                format=LocalDecision.model_json_schema(),
                think=False,
                options={'num_ctx': 8192,
                         'num_predict': 2048,
                         'temperature': 0.3 * attempt},
            )
            # Validation sits inside the try so cut-off JSON triggers a retry.
            answer = LocalDecision.model_validate_json(response.message.content)
            break
        except (httpx.ReadTimeout, ValidationError):
            print(f"Failed on {claim['claim_id']} (attempt {attempt + 1})")
            local.generate(model=model, keep_alive=0)  # reset a stuck runner; the retry reloads it
            if attempt == tries - 1:
                raise
    return {
        'model': model,
        'effort': 'none',
        'where': 'This computer',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'input_tokens': response.prompt_eval_count,
        'cost': 0.0,
        'seconds': round(time.time() - start, 1),
    }

In [27]:
def run_locally(model, claims, labels):
    """Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim['claim_id']]))
    local.generate(model=model, keep_alive=0)  # unload the model, so the next one has the memory
    report(results)
    return results

In [28]:
# Smoke test one local call before running all 20 claims.
first_local = decide_locally(claims[0], 'qwen3.5:4b')
first_local

{'model': 'qwen3.5:4b',
 'effort': 'none',
 'where': 'This computer',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim is for Trip Delay. The policy record shows the plan is 'premium'. The incident date (2026-06-18) falls within the trip dates (departs 2026-06-10, returns 2026-06-18). The summary states the flight was canceled due to a 'crew problem', which is a covered cause under the Trip Delay section. The claim includes accommodation, meals, and transport, which are covered expenses. All required documents (carrier report and receipts) are attached. \n\nCalculation:\n1. Plan: Premium.\n2. Trip Delay Maximum for Premium: $800.\n3. Items claimed: Hotel ($245), Meals ($95), Transport ($80). Total = $420.\n4. Deductible for Premium is $50, but this applies to Baggage and Emergency Medical only, not Trip Delay. So deductible is $0.\n5. Payment calculation: Min($420, $800) - $0 = $420.\n6. The result ($420) is less than the maximum ($800), so th

In [29]:
# Qwen 3.5 4B: the smallest model here. Its decision score is lower than the others'.
qwen = run_locally('qwen3.5:4b', claims, labels)

qwen3.5:4b:   0%|          | 0/20 [00:00<?, ?it/s]

Failed on CLM-1007 (attempt 1)
qwen3.5:4b: 13 of 20 decisions right, 7 with the payout too, $0.0000 in 169 seconds


In [30]:
# Gemma 4 E4B: the stronger of the two Gemma models.
gemma_e4b = run_locally('gemma4:e4b', claims, labels)

gemma4:e4b:   0%|          | 0/20 [00:00<?, ?it/s]

gemma4:e4b: 15 of 20 decisions right, 10 with the payout too, $0.0000 in 101 seconds


In [31]:
# Gemma 4 E2B is the smaller, quantized version. Pull it before the first run.
print(ollama.pull('gemma4:e2b-it-qat').status)

success


In [32]:
# Gemma 4 E2B: fastest of the local models, but weakest on payouts.
gemma_e2b = run_locally('gemma4:e2b-it-qat', claims, labels)

gemma4:e2b-it-qat:   0%|          | 0/20 [00:00<?, ?it/s]

gemma4:e2b-it-qat: 15 of 20 decisions right, 7 with the payout too, $0.0000 in 66 seconds


### Optional: a tenth model direct on Cerebras

OpenRouter passes each request to a provider. You can also call a provider directly with your own account. Cerebras builds its own inference chips and serves Qwen 3.8 27B very quickly. This section needs `CEREBRAS_API_KEY` in `.env`. If you skip it, remove `qwen_cerebras`, `qwen_cerebras_low` and `qwen_cerebras_high` from the `experiments` list in Part 4.

Cerebras does not report the cost of a call, so it is worked out from the token counts and the listed prices: $0.99 per million input tokens and $1.49 per million output tokens (28 September 2026). The model supports four reasoning levels, `none`, `low`, `medium` and `high`; this notebook runs `none`, `low` and `high`.

In [33]:
# Read the key and build a client pointed at Cerebras, using the same OpenAI client as before.
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
CEREBRAS_BASE_URL = "https://api.cerebras.ai/v1"

if cerebras_api_key:
    print("Cerebras API key found")
    cerebras = OpenAI(base_url=CEREBRAS_BASE_URL, api_key=cerebras_api_key, timeout=120, max_retries=5)
else:
    print("No Cerebras API key found. Add CEREBRAS_API_KEY to the .env file")

Cerebras API key found


In [34]:
CEREBRAS_INPUT_PRICE = 0.99 / 1_000_000
CEREBRAS_OUTPUT_PRICE = 1.49 / 1_000_000


def decide_on_cerebras(claim, model, effort):
    """Ask a model on Cerebras to decide one claim."""
    start = time.time()
    response = cerebras.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
    )
    answer = response.choices[0].message.parsed
    usage = response.usage
    return {
        'model': model,
        'effort': effort,
        'where': 'Cerebras',
        'claim_id': claim['claim_id'],
        'decision': answer.decision,
        'payout': answer.payout,
        'rationale': answer.rationale,
        'input_tokens': usage.prompt_tokens,
        # Cost is computed from token counts, because the API does not report it.
        'cost': usage.prompt_tokens * CEREBRAS_INPUT_PRICE + usage.completion_tokens * CEREBRAS_OUTPUT_PRICE,
        'seconds': round(time.time() - start, 1),
    }


def run_on_cerebras(model, effort, claims, labels):
    """Run a model on Cerebras on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_on_cerebras(claim, model, effort)
        results.append(score(answer, labels[claim['claim_id']]))
    report(results)
    return results

In [35]:
# Reasoning "none": the fastest and cheapest setting.
qwen_cerebras = run_on_cerebras('qwen-3.8-27b', 'none', claims, labels)

qwen-3.8-27b:   0%|          | 0/20 [00:00<?, ?it/s]

qwen-3.8-27b: 14 of 20 decisions right, 10 with the payout too, $0.0322 in 7 seconds


In [36]:
# Reasoning "low".
qwen_cerebras_low = run_on_cerebras('qwen-3.8-27b', 'low', claims, labels)

qwen-3.8-27b:   0%|          | 0/20 [00:00<?, ?it/s]

qwen-3.8-27b: 17 of 20 decisions right, 15 with the payout too, $0.0579 in 18 seconds


In [37]:
# Reasoning "high": more thinking per claim, so a higher cost and a longer run.
qwen_cerebras_high = run_on_cerebras('qwen-3.8-27b', 'high', claims, labels)

qwen-3.8-27b:   0%|          | 0/20 [00:00<?, ?it/s]

qwen-3.8-27b: 16 of 20 decisions right, 11 with the payout too, $0.0837 in 40 seconds


### Part 3 summary

You ran three small models on your own computer and one model direct on Cerebras at three reasoning levels. The local runs are free but slow, and they exposed failure modes the hosted models hid: timeouts, cut-off JSON and a missing `payout` field. `decide_locally` retries those, and `LocalDecision` reads a missing payout as 0.

In the saved run, Gemma E4B matched GLM at 11 of 20 right with the payout, while Qwen 4B reached 7 and Gemma E2B only 4. Gemma E2B's decisions were often right (13 of 20) but its payouts rarely were, partly because a missing payout is scored as 0. Cerebras ran 20 claims in 11 to 56 seconds. Low reasoning scored best (14 of 20 with the payout), and high reasoning cost more without scoring higher.

## Part 4: Accuracy, cost and speed side by side

Collect every run into one table with one row per model and reasoning level, then chart accuracy against cost, accuracy against speed, and all three together. Each point is one run, coloured by where it ran: OpenRouter, Cerebras or this computer.

In [39]:
# If you skipped a section, take its runs out of this list.
# qwen_cerebras* need the Cerebras key; qwen, gemma_e2b and gemma_e4b need Ollama.
experiments = [mimo,
               glm,
               kimi,
               deepseek,
               nemotron,
               mistral,
               qwen,
               gemma_e4b,
               gemma_e2b,
               qwen_cerebras,
               qwen_cerebras_low,
               qwen_cerebras_high,
               ]

# One row per run: percentages of claims right, and the average time and cost per claim.
rows = []
for results in experiments:
    count = len(results)
    rows.append({
        'model': results[0]['model'].split('/')[-1],
        'where': results[0]['where'],
        'reasoning': results[0]['effort'] or 'not set',
        'decision right (%)': round(100 * sum(result['decision_right'] for result in results) / count),
        'decision and payout right (%)': round(100 * sum(result['right'] for result in results) / count),
        'seconds a claim': round(sum(result['seconds'] for result in results) / count, 1),
        'cost a claim ($)': round(sum(result['cost'] for result in results) / count, 5),
    })

summary = pd.DataFrame(rows)
summary

,model,where,reasoning,decision right (%),decision and payout right (%),seconds a claim,cost a claim ($)
0,mimo-v2.6-pro,OpenRouter,not set,85,75,35.0,0.00111
1,glm-5.3-flash,OpenRouter,low,70,55,8.6,0.00020
2,kimi-k3,OpenRouter,low,80,70,5.8,0.00722
3,deepseek-v4.1-flash,OpenRouter,low,85,80,7.4,0.00094
4,nemotron-3-ultra-550b-a55b,OpenRouter,medium,70,55,4.4,0.00114
5,mistral-medium-3-5,OpenRouter,none,75,55,1.7,0.00265
6,qwen3.5:4b,This computer,none,65,35,8.5,0.00000
7,gemma4:e4b,This computer,none,75,50,5.1,0.00000
8,gemma4:e2b-it-qat,This computer,none,75,35,3.3,0.00000
9,qwen-3.8-27b,Cerebras,none,70,50,0.3,0.00161


### Accuracy against cost

Points higher on the chart are more accurate, and points further left are cheaper. Local models sit at zero cost. The best runs are up and to the left.

In [41]:
fig = px.scatter(
    summary,
    x='cost a claim ($)',
    y='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy against cost',
    height=600,
)
fig.update_traces(marker_size=14, textposition='top center')
fig.update_yaxes(range=[0, 100])
fig.show()

### Accuracy against speed

Points further left take less time per claim. This is the chart where Cerebras stands apart from the other providers.

In [42]:
fig = px.scatter(
    summary,
    x='seconds a claim',
    y='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy against speed',
    height=600,
)
fig.update_traces(marker_size=14, textposition='top right')
fig.update_yaxes(range=[0, 100])
fig.show()

### All three at once

Cost, time and accuracy in one chart. Drag to rotate it. The best runs are high on the vertical axis and near the origin of the other two.

In [43]:
fig = px.scatter_3d(
    summary,
    x='cost a claim ($)',
    y='seconds a claim',
    z='decision and payout right (%)',
    color='where',
    text='model',
    title='Accuracy, cost and speed',
    height=700,
)
fig.show()

### Part 4 summary

The table and charts put twelve runs on one scale. No run wins on all three measures: the most accurate models cost more, the fastest ones are less accurate, and the free local models pay in time and in missed payouts. Treat each point as a measurement from one pass over 20 claims, not a ranking. One claim is worth five percentage points.

## Conclusion

This notebook compares open-weight models on UnderCover's labelled claims using one prompt, one schema and one scoring rule. It shows that sending only the wording a claim needs cuts the input by more than half, that hosted, local and direct-provider models can all be driven by the same code, and that a run should be judged on accuracy, cost and speed together.

In the saved run, Kimi and DeepSeek tied for the best result with the payout (16 of 20), and DeepSeek was about ten times cheaper. Cerebras was much the fastest hosted option, and low reasoning beat high. Gemma E4B was the best free option, and the smallest local models struggled with payouts.

The results are limited to 20 claims, one prompt, the model versions and provider routing at the time, and a single run each. They do not prove production readiness or parity with handlers. Before relying on a model, rerun it from a clean kernel, repeat it to see how much it varies, inspect its misses, and test it on the held-out claims. Handler labels must stay out of model context throughout.